In [21]:
from pathlib import Path
import duckdb

BASE_DIR = Path().resolve().parent
db_path = BASE_DIR / "database" / "citibike.duckdb"

con = duckdb.connect(str(db_path))

%reload_ext sql
%sql con
%config SqlMagic.autolimit = 50

%config SqlMagic.displaylimit = 15

In [22]:
%%sql
SELECT
    COUNT(*) AS total,
    COUNT(*) FILTER (WHERE ended_at <= started_at) AS inverted_or_zero,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) BETWEEN 1 AND 59) AS under_1_min,
    COUNT(*) FILTER (WHERE DATE_DIFF('second', started_at, ended_at) > 86400) AS over_24h,
    COUNT(*) FILTER (WHERE start_station_id IS NULL) AS no_start_station,
    COUNT(*) FILTER (WHERE end_station_id IS NULL) AS no_end_station,
    COUNT(*) - COUNT(DISTINCT ride_id) AS duplicated_rides
FROM trips_raw;

Running query in 'DuckDBPyConnection'

total,inverted_or_zero,under_1_min,over_24h,no_start_station,no_end_station,duplicated_rides
4982855,0,0,1070,2917,14576,0


In [23]:
%%sql
/*
trips_clean: filtered view over trips_raw, used by all analysis queries.

Removes:
- ended_at <= started_at: a trip cannot end before it starts, so these are
  system or recording errors.

- duration < 60 seconds: usually false starts (bike undocked and returned
  right away) and not real trips.
  
- duration > 24 hours (86,400 s): usually lost, forgotten or unreturned bikes.

The raw data stays untouched in trips_raw. Row counts removed by each rule
are listed in the README.
*/
CREATE OR REPLACE VIEW trips_clean AS (
    SELECT
        * EXCLUDE (filename),
        DATE_DIFF('second', started_at, ended_at) AS duration_seconds
    FROM trips_raw
    WHERE ended_at > started_at
        AND DATE_DIFF('second', started_at, ended_at) >= 60
        AND DATE_DIFF('second', started_at, ended_at) <= 86400
);

Running query in 'DuckDBPyConnection'

Count


In [24]:
%%sql
CREATE OR REPLACE TABLE stations AS
WITH all_points AS (
    SELECT 
        start_station_id AS station_id, 
        start_station_name AS station_name,
        start_lat AS lat, 
        start_lng AS lng
    FROM trips_raw
    UNION ALL
    SELECT end_station_id, end_station_name, end_lat, end_lng
    FROM trips_raw
)
SELECT
    station_id,
    MODE(station_name) AS station_name,
    AVG(lat) AS lat,
    AVG(lng) AS lng,
    COUNT(*) AS trip_ends
FROM all_points
WHERE station_id IS NOT NULL
GROUP BY station_id;

Running query in 'DuckDBPyConnection'

Count
2501


In [25]:
%%sql
-- HOW MANY LINES DOES TRIPS_RAW AND TRIPS_CLEAN HAVE? AND HOW MANY LINES WERE REMOVED FOR EACH RULE?
SELECT
    (SELECT COUNT(*) FROM trips_raw) AS raw_rows,
    (SELECT COUNT(*) FROM trips_clean) AS clean_rows,
    ((SELECT COUNT(*) FROM trips_raw) - (SELECT COUNT(*) FROM trips_clean)) AS lines_removed;


Running query in 'DuckDBPyConnection'

raw_rows,clean_rows,lines_removed
4982855,4981785,1070


In [26]:
%%sql
--WHAT ARE THE MIN AND THE MAX DATES OF STARTED_AT?
SELECT
    MIN(started_at),
    MAX(started_at)
FROM trips_clean;

Running query in 'DuckDBPyConnection'

min(started_at),max(started_at)
2026-08-31 10:13:26.230000,2026-09-30 23:58:16.821000


In [27]:
%%sql
--how many stations have null values?
SELECT
    COUNT(*) FILTER(WHERE start_station_name IS NULL),
    COUNT(*) FILTER(WHERE end_station_name	IS NULL)
FROM trips_clean;

Running query in 'DuckDBPyConnection'

count_star() FILTER (WHERE (start_station_name IS NULL)),count_star() FILTER (WHERE (end_station_name IS NULL))
2917,12404


In [28]:
%%sql
--HOW MANY TRAVELS HAVE NULL STATIONS AND IN WHICH RIDEABLE TYPE HAPPENS MORE?

SELECT
    rideable_type,
    COUNT(*) FILTER(WHERE start_station_name IS NULL) AS started_stations,
    COUNT(*) FILTER(WHERE end_station_name IS NULL) AS ended_station
FROM trips_clean
GROUP BY rideable_type;

Running query in 'DuckDBPyConnection'

rideable_type,started_stations,ended_station
electric_bike,2917,12338
classic_bike,0,66


In [55]:
%%sql
/************************************************************************************************
For each station, what is the net flow (bikes arriving minus bikes departing) over the full day? 
Which stations are consistently "bike deficits" and which are "bike surpluses"?
************************************************************************************************/

WITH departing AS (
    SELECT
        start_station_name,
        COUNT(started_at) AS bikes_departing
    FROM trips_clean
    GROUP BY start_station_name
),

arriving AS (
    SELECT
        end_station_name,
        COUNT(ended_at) AS bikes_arriving
    FROM trips_clean
    GROUP BY end_station_name
)

SELECT
    COALESCE(start_station_name, end_station_name, 'unknown') AS station,
    COALESCE(bikes_departing, 0) AS bikes_departing,
    COALESCE(bikes_arriving, 0) AS bikes_arriving,
    COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) AS net_flow,
    CASE WHEN COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) < 0 THEN 'bike deficits' 
        WHEN COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) > 0 THEN 'bike surpluses'
        ELSE 'regular' END AS classification
FROM departing d
FULL JOIN arriving a
    ON d.start_station_name = a.end_station_name
ORDER BY net_flow ASC;

Running query in 'DuckDBPyConnection'

station,bikes_departing,bikes_arriving,net_flow,classification
unknown,2917,0,-2917,bike deficits
E 39 St & Lexington Ave,7489,6849,-640,bike deficits
E 33 St & Park Ave,4165,3552,-613,bike deficits
Willoughby Ave & Hall St,5593,5100,-493,bike deficits
E 39 St & 2 Ave,5768,5276,-492,bike deficits
E 51 St & 2 Ave,5726,5248,-478,bike deficits
W 45 St & 8 Ave,8409,8058,-351,bike deficits
E 35 St & Madison Ave,5103,4757,-346,bike deficits
6 Ave & W 34 St,8856,8531,-325,bike deficits
46 St & Queens Blvd,1810,1489,-321,bike deficits


## What are the busiest hours of the day for bike departures, overall and broken down by weekday vs. weekend?


In [30]:
%%sql
--First part: What are the busiest hours of the day for bikes_departures, overall.

SELECT
    EXTRACT(HOUR FROM started_at) AS DayOfDay,
    COUNT(*) AS bikes_departing
FROM trips_clean
WHERE started_at IS NOT NULL
GROUP BY DayOfDay
ORDER BY bikes_departing DESC;

Running query in 'DuckDBPyConnection'

DayOfDay,bikes_departing
17,484123
18,453018
16,371007
19,345305
15,318174
8,317936
14,290315
13,268359
9,255187
12,246514


In [31]:
%%sql

--second part: What are the busiest hours of the day for bikes_departures, brokwn down by weeldays vs weekend.
SELECT
    CASE
        WHEN EXTRACT(ISODOW FROM started_at) BETWEEN 1 AND 5 THEN 'weekdays'
        ELSE 'weekend'
    END AS day_type,
    EXTRACT(HOUR FROM started_at) AS hourOfDay,
    COUNT(*) AS bikes_departing
FROM trips_clean
WHERE started_at IS NOT NULL
GROUP BY EXTRACT(HOUR FROM started_at), day_type
ORDER BY bikes_departing DESC


Running query in 'DuckDBPyConnection'

day_type,hourOfDay,bikes_departing
weekdays,17,405146
weekdays,18,376382
weekdays,8,295280
weekdays,16,293076
weekdays,19,280509
weekdays,15,238117
weekdays,9,219573
weekdays,14,211913
weekdays,20,193904
weekdays,13,192137


>We can see a peek of departing from the hours 17 and 18. mainly caused due to the work comute, home <> work and mainly on weekdays not in weekends.

## Which 10 stations have the highest total trip volume (start + end trips combined)?

In [32]:
%%sql
SELECT
    station,
    COUNT(*) AS total_trips_volume
FROM (
    SELECT start_station_name AS station
    FROM trips_clean

    UNION ALL

    SELECT end_station_name AS station
    FROM trips_clean
)
GROUP BY station
ORDER BY total_trips_volume DESC
LIMIT 10;

Running query in 'DuckDBPyConnection'

station,total_trips_volume
W 21 St & 6 Ave,34636
Pier 61 at Chelsea Piers,33762
E 17 St & Broadway,30583
W 31 St & 7 Ave,30482
9 Ave & W 33 St,30019
Broadway & W 58 St,29602
Park Ave & E 42 St,29176
University Pl & E 14 St,27546
Broadway & E 14 St,27242
Broadway & W 25 St,26512


## Is there a difference in trip duration or patterns between "member" and "casual" riders?

In [33]:
%%sql
SELECT
    COALESCE(member_casual, 'unknown'),
    COUNT(rideable_type) bike_type,
    AVG(COALESCE(duration_seconds, 0)) / 60 AS duration
FROM trips_clean
GROUP BY member_casual

Running query in 'DuckDBPyConnection'

"COALESCE(member_casual, 'unknown')",bike_type,duration
casual,934740,18.470183616121417
member,4047045,11.891783560597919


In [34]:
%%sql
SELECT
    COALESCE(member_casual, 'unknown') AS member_type,
    COALESCE(rideable_type, 'unknown') bike_type,
    COUNT(rideable_type) AS quantity,
    AVG(COALESCE(duration_seconds, 0)) / 60 AS duration
FROM trips_clean
GROUP BY member_casual, rideable_type
ORDER BY quantity DESC, duration DESC;

Running query in 'DuckDBPyConnection'

member_type,bike_type,quantity,duration
member,electric_bike,2899274,11.979066793962904
member,classic_bike,1147771,11.671305774409705
casual,electric_bike,719535,17.414312090447304
casual,classic_bike,215205,22.00047574793027


> Both casual or member riders prefer to trip with eletric bikes with a huge gap between the classic bikes but casual riders have an avg duration higher than the member riders.

## What is the average trip duration, and are there suspicious outliers (e.g., trips lasting 0 seconds or several days) that suggest bad data you'd need to filter out?

In [35]:
%%sql
--What us the average trip duration?
SELECT
    AVG(COALESCE(duration_seconds, 0) / 60) as avg_duration
FROM trips_clean

Running query in 'DuckDBPyConnection'

avg_duration
13.126098904977527


In [36]:
%%sql
--are there suspicius outliers?
SELECT
    COUNT(*) FILTER(WHERE duration_seconds = 0) AS trips_lasting_0,
    COUNT(*) FILTER(WHERE DATE_DIFF('seconds', started_at, ended_at) > 86400) trips_lasting_days
FROM trips_clean;

Running query in 'DuckDBPyConnection'

trips_lasting_0,trips_lasting_days
0,0


>before we started we checked our dataset to see is there were any trip the lasted more than 24 hour or less than a minute. And there were, so we removed all of them, so there´s no outlier to mess up with out analysis

## Which start–end station pairs are the most common routes? Does this suggest a commuter pattern (same stations, weekday mornings) or leisure pattern (weekend, varied stations)?

In [37]:
%%sql
SELECT
    start_station_name,
    end_station_name,
    COUNT(*) AS total_trips
FROM trips_clean
GROUP BY start_station_name, end_station_name
ORDER BY total_trips DESC
LIMIT 20;

Running query in 'DuckDBPyConnection'

start_station_name,end_station_name,total_trips
Central Park S & 6 Ave,Central Park S & 6 Ave,2138
7 Ave & Central Park South,7 Ave & Central Park South,1310
55 Ave & Center Blvd,Vernon Blvd & 50 Ave,987
Vernon Blvd & 50 Ave,55 Ave & Center Blvd,917
Roosevelt Island Tramway,Roosevelt Island Tramway,846
S 4 St & Wythe Ave,N 6 St & Bedford Ave,836
Soissons Landing,Soissons Landing,830
Eastern Pkwy & Kingston Ave,Eastern Pkwy & Kingston Ave,814
N 6 St & Bedford Ave,S 2 St & Kent Ave,810
N 6 St & Bedford Ave,S 4 St & Wythe Ave,781


In [41]:
%%sql
SELECT
    start_station_name,
    end_station_name,
    EXTRACT(HOUR FROM started_at) AS hour,
    COUNT(*) AS trips
FROM trips_clean
GROUP BY 1,2,3
ORDER BY trips DESC;

Running query in 'DuckDBPyConnection'

start_station_name,end_station_name,hour,trips
Central Park S & 6 Ave,Central Park S & 6 Ave,14,246
Central Park S & 6 Ave,Central Park S & 6 Ave,17,233
Central Park S & 6 Ave,Central Park S & 6 Ave,15,230
Central Park S & 6 Ave,Central Park S & 6 Ave,13,209
Central Park S & 6 Ave,Central Park S & 6 Ave,11,203
Central Park S & 6 Ave,Central Park S & 6 Ave,12,198
Central Park S & 6 Ave,Central Park S & 6 Ave,16,182
North Moore St & Greenwich St,Barclay St & Church St,16,180
North Moore St & Greenwich St,Vesey Pl & River Terrace,17,160
N 6 St & Bedford Ave,S 2 St & Kent Ave,9,155


In [39]:
%%sql
SELECT
    member_casual,
    start_station_name,
    end_station_name,
    COUNT(*) AS trips
FROM trips_clean
GROUP BY 1,2,3
ORDER BY trips DESC;

Running query in 'DuckDBPyConnection'

member_casual,start_station_name,end_station_name,trips
casual,Central Park S & 6 Ave,Central Park S & 6 Ave,1874
casual,7 Ave & Central Park South,7 Ave & Central Park South,1184
member,55 Ave & Center Blvd,Vernon Blvd & 50 Ave,916
member,Vernon Blvd & 50 Ave,55 Ave & Center Blvd,867
member,Eastern Pkwy & Kingston Ave,Eastern Pkwy & Kingston Ave,764
member,N 6 St & Bedford Ave,S 2 St & Kent Ave,741
member,S 4 St & Wythe Ave,N 6 St & Bedford Ave,735
casual,Roosevelt Island Tramway,Roosevelt Island Tramway,681
member,N 6 St & Bedford Ave,S 4 St & Wythe Ave,679
casual,Soissons Landing,Soissons Landing,640


## If you had to recommend 5 stations for overnight bike rebalancing (moving bikes in), which would they be, and what data backs that recommendation?

In [ ]:
%%sql
--first we need our net flow to answer that question
WITH bikes_departing AS (
    SELECT
        start_station_name,
        COUNT(*) AS departing_quantity
    FROM trips_clean
    GROUP BY start_station_name
),
bikes_arriving AS (
    SELECT
        end_station_name,
        COUNT(*) AS arriving_quantity
    FROM trips_clean
    GROUP BY end_station_name
)

SELECT
    COALESCE(start_station_name, end_station_name, 'unknown') AS station,
    COALESCE(arriving_quantity, 0) AS bikes_arriving,
    COALESCE(departing_quantity, 0) AS bikes_departing,
    (COALESCE(arriving_quantity) - COALESCE(departing_quantity, 0)) AS net_flow
FROM bikes_departing bd
FULL JOIN bikes_arriving ba
    ON bd.start_station_name = ba.end_station_name
ORDER BY net_flow ASC;


Running query in 'DuckDBPyConnection'

station,bikes_arriving,bikes_departing,net_flow
E 39 St & Lexington Ave,6849,7489,-640
E 33 St & Park Ave,3552,4165,-613
Willoughby Ave & Hall St,5100,5593,-493
E 39 St & 2 Ave,5276,5768,-492
E 51 St & 2 Ave,5248,5726,-478
W 45 St & 8 Ave,8058,8409,-351
E 35 St & Madison Ave,4757,5103,-346
6 Ave & W 34 St,8531,8856,-325
46 St & Queens Blvd,1489,1810,-321
8 Ave & W 33 St,8485,8800,-315


: 

In [ ]:
%%sql
/************************************************************************************************
For each station, what is the net flow (bikes arriving minus bikes departing) over the full day? 
Which stations are consistently "bike deficits" and which are "bike surpluses"?
************************************************************************************************/

WITH departing AS (
    SELECT
        start_station_name,
        COUNT(started_at) AS bikes_departing
    FROM trips_clean
    GROUP BY start_station_name
),

arriving AS (
    SELECT
        end_station_name,
        COUNT(ended_at) AS bikes_arriving
    FROM trips_clean
    GROUP BY end_station_name
)

SELECT
    COALESCE(start_station_name, end_station_name, 'unknown') AS station,
    COALESCE(bikes_departing, 0) AS bikes_departing,
    COALESCE(bikes_arriving, 0) AS bikes_arriving,
    COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) AS net_flow,
    CASE WHEN COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) < 0 THEN 'bike deficits' 
        WHEN COALESCE(a.bikes_arriving, 0) - COALESCE(d.bikes_departing, 0) > 0 THEN 'bike surpluses'
        ELSE 'regular' END AS classification
FROM departing d
FULL JOIN arriving a
    ON d.start_station_name = a.end_station_name
ORDER BY net_flow ASC;

Running query in 'DuckDBPyConnection'

station,bikes_departing,bikes_arriving,net_flow,classification
unknown,2917,0,-2917,bike deficits
E 39 St & Lexington Ave,7489,6849,-640,bike deficits
E 33 St & Park Ave,4165,3552,-613,bike deficits
Willoughby Ave & Hall St,5593,5100,-493,bike deficits
E 39 St & 2 Ave,5768,5276,-492,bike deficits
E 51 St & 2 Ave,5726,5248,-478,bike deficits
W 45 St & 8 Ave,8409,8058,-351,bike deficits
E 35 St & Madison Ave,5103,4757,-346,bike deficits
6 Ave & W 34 St,8856,8531,-325,bike deficits
46 St & Queens Blvd,1810,1489,-321,bike deficits
